In [0]:
# ===========================================
# Notebook Name:
# 05_build_deck_cards
#
# Purpose:
# Parse raw Pokemon deck HTML and build the
# card-level Silver table. Parsing logic is
# intentionally duplicated from
# 04_build_decks, since each notebook is a
# distinct task in the Silver job DAG and
# cannot share in-memory state.
#
# Input:
# pokemon.bronze.deck_raw
# pokemon.silver.decks
#
# Output:
# pokemon.silver.deck_cards
#
# Natural key:
# deck_code, category, card_name,
# expansion, collection_number
# (nullable columns make a MERGE match
# unreliable under SQL NULL semantics, so
# this notebook uses delete-then-insert per
# deck_code instead of MERGE)
#
# Change detection:
# response_hash
# ===========================================

In [0]:
%pip install beautifulsoup4

In [0]:
dbutils.library.restartPython()

In [0]:
import re
from typing import Any

from bs4 import BeautifulSoup

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window


DECK_RAW_TABLE = "pokemon.bronze.deck_raw"
DECKS_TABLE = "pokemon.silver.decks"
DECK_CARDS_TABLE = (
    "pokemon.silver.deck_cards"
)

print("Input :", DECK_RAW_TABLE)
print("Input :", DECKS_TABLE)
print("Output:", DECK_CARDS_TABLE)

In [0]:
CATEGORY_NAMES = {
    "ポケモン": "pokemon",
    "グッズ": "goods",
    "ポケモンのどうぐ": "pokemon_tool",
    "サポート": "supporter",
    "スタジアム": "stadium",
    "エネルギー": "energy",
}


def normalize_text(value: str | None) -> str:
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        value.replace("\u3000", " "),
    ).strip()


def parse_quantity(value: str) -> int | None:
    normalized = normalize_text(value)

    if re.fullmatch(r"\d+", normalized):
        return int(normalized)

    match = re.fullmatch(
        r"(\d+)\s*枚",
        normalized,
    )

    if match:
        return int(match.group(1))

    return None

In [0]:
def parse_deck_html(
    html: str,
    deck_code: str,
) -> list[dict[str, Any]]:

    if not html:
        return []

    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    cards: list[dict[str, Any]] = []
    current_category: str | None = None

    for row in soup.find_all("tr"):
        row_cells = [
            normalize_text(
                cell.get_text(
                    " ",
                    strip=True,
                )
            )
            for cell in row.find_all(
                ["th", "td"],
                recursive=False,
            )
        ]

        row_cells = [
            cell
            for cell in row_cells
            if cell
        ]

        if not row_cells:
            continue

        first_cell = row_cells[0]

        if first_cell in CATEGORY_NAMES:
            current_category = (
                CATEGORY_NAMES[first_cell]
            )
            continue

        if current_category is None:
            continue

        if any(
            keyword in first_cell
            for keyword in [
                "カード名",
                "小計",
                "合計",
            ]
        ):
            continue

        quantity: int | None = None
        quantity_index: int | None = None

        for index, cell in enumerate(
            row_cells[1:],
            start=1,
        ):
            parsed_quantity = (
                parse_quantity(cell)
            )

            if parsed_quantity is not None:
                quantity = parsed_quantity
                quantity_index = index
                break

        if (
            quantity is None
            or quantity_index is None
        ):
            continue

        remaining_cells = row_cells[
            quantity_index + 1:
        ]

        expansion = (
            remaining_cells[0]
            if len(remaining_cells) >= 1
            else None
        )

        collection_number = (
            remaining_cells[1]
            if len(remaining_cells) >= 2
            else None
        )

        cards.append({
            "deck_code": deck_code,
            "category": current_category,
            "card_name": first_cell,
            "quantity": quantity,
            "expansion": expansion,
            "collection_number": (
                collection_number
            ),
        })

    return cards

In [0]:
latest_window = (
    Window
    .partitionBy("deck_code")
    .orderBy(
        F.col("scraped_at").desc(),
        F.col("response_hash").desc(),
    )
)

latest_deck_raw_df = (
    spark.table(DECK_RAW_TABLE)
    .withColumn(
        "_row_number",
        F.row_number().over(
            latest_window
        ),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

if spark.catalog.tableExists(
    DECK_CARDS_TABLE
):
    existing_cards_hash_df = (
        spark.table(DECK_CARDS_TABLE)
        .select(
            "deck_code",
            "response_hash",
        )
        .distinct()
        .withColumnRenamed(
            "response_hash",
            "existing_response_hash",
        )
    )

else:
    existing_cards_hash_df = (
        spark.createDataFrame(
            [],
            """
            deck_code STRING,
            existing_response_hash STRING
            """,
        )
    )

changed_deck_raw_df = (
    latest_deck_raw_df.alias("raw")
    .join(
        existing_cards_hash_df.alias(
            "existing"
        ),
        on="deck_code",
        how="left",
    )
    .filter(
        F.col(
            "existing_response_hash"
        ).isNull()
        | (
            F.col(
                "existing_response_hash"
            )
            != F.col("response_hash")
        )
    )
    .select("raw.*")
)

print(
    "Latest deck count:",
    latest_deck_raw_df.count(),
)

print(
    "New/changed deck count:",
    changed_deck_raw_df.count(),
)

In [0]:
raw_deck_rows = (
    changed_deck_raw_df
    .select(
        "deck_code",
        "source_url",
        "response_hash",
        "scraped_at",
        "payload",
    )
    .collect()
)

card_rows: list[dict[str, Any]] = []
parse_errors: list[dict[str, str]] = []

for index, row in enumerate(
    raw_deck_rows,
    start=1,
):
    deck_code = row["deck_code"]

    print(
        f"[{index}/{len(raw_deck_rows)}] "
        f"Parsing {deck_code}"
    )

    try:
        cards = parse_deck_html(
            html=row["payload"],
            deck_code=deck_code,
        )

        if not cards:
            raise ValueError(
                "No cards parsed from "
                "deck HTML"
            )

        for card in cards:
            card_rows.append({
                "deck_code": deck_code,
                "category": card[
                    "category"
                ],
                "card_name": card[
                    "card_name"
                ],
                "quantity": card[
                    "quantity"
                ],
                "expansion": card[
                    "expansion"
                ],
                "collection_number": (
                    card[
                        "collection_number"
                    ]
                ),
                "source_url": (
                    row["source_url"]
                ),
                "response_hash": (
                    row["response_hash"]
                ),
                "source_scraped_at": (
                    row["scraped_at"]
                ),
            })

    except Exception as error:
        parse_errors.append({
            "deck_code": deck_code,
            "error": str(error),
        })

        print("  ERROR:", error)

print(
    "Parsed decks:",
    len(raw_deck_rows)
    - len(parse_errors),
)
print("Parsed card rows:", len(card_rows))
print("Errors:", len(parse_errors))

if parse_errors:
    for error in parse_errors:
        print(error)

    raise ValueError(
        f"{len(parse_errors)} deck "
        "parsing errors occurred"
    )

In [0]:
card_schema = StructType([
    StructField(
        "deck_code",
        StringType(),
        False,
    ),
    StructField(
        "category",
        StringType(),
        False,
    ),
    StructField(
        "card_name",
        StringType(),
        False,
    ),
    StructField(
        "quantity",
        IntegerType(),
        False,
    ),
    StructField(
        "expansion",
        StringType(),
        True,
    ),
    StructField(
        "collection_number",
        StringType(),
        True,
    ),
    StructField(
        "source_url",
        StringType(),
        True,
    ),
    StructField(
        "response_hash",
        StringType(),
        True,
    ),
    StructField(
        "source_scraped_at",
        TimestampType(),
        True,
    ),
])

deck_cards_staged_df = (
    spark.createDataFrame(
        card_rows,
        schema=card_schema,
    )
)

print(
    "Staged card rows:",
    deck_cards_staged_df.count(),
)

In [0]:
deck_hash_lookup_df = (
    spark.table(DECKS_TABLE)
    .select("deck_code", "deck_hash")
)

deck_cards_df = (
    deck_cards_staged_df
    .join(
        deck_hash_lookup_df,
        on="deck_code",
        how="left",
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
    .select(
        "deck_code",
        "deck_hash",
        "category",
        "card_name",
        "quantity",
        "expansion",
        "collection_number",
        "source_url",
        "response_hash",
        "source_scraped_at",
        "updated_at",
    )
)

missing_hash_df = (
    deck_cards_df.filter(
        F.col("deck_hash").isNull()
    )
)

missing_hash_count = (
    missing_hash_df
    .select("deck_code")
    .distinct()
    .count()
)

if missing_hash_count > 0:
    display(
        missing_hash_df
        .select("deck_code")
        .distinct()
    )

    raise ValueError(
        f"{missing_hash_count} deck_codes "
        f"have no matching row in "
        f"{DECKS_TABLE}. Run "
        "04_build_decks first."
    )

print(
    "Validation passed: "
    "all deck_codes resolved to a "
    "deck_hash"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{DECK_CARDS_TABLE}
(
    deck_code STRING NOT NULL,
    deck_hash STRING,
    category STRING NOT NULL,
    card_name STRING NOT NULL,
    quantity INT NOT NULL,
    expansion STRING,
    collection_number STRING,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT
'Normalized Pokemon deck card-level rows'
""")

In [0]:
changed_deck_codes = [
    row["deck_code"]
    for row in (
        deck_cards_df
        .select("deck_code")
        .distinct()
        .collect()
    )
]

print(
    "Deck codes to replace:",
    len(changed_deck_codes),
)

if changed_deck_codes:
    deck_cards_df.createOrReplaceTempView(
        "staged_deck_cards"
    )

    deck_codes_literal = ", ".join(
        "'{}'".format(
            code.replace("'", "''")
        )
        for code in changed_deck_codes
    )

    spark.sql(f"""
    DELETE FROM {DECK_CARDS_TABLE}
    WHERE deck_code IN
    ({deck_codes_literal})
    """)

    (
        spark.table("staged_deck_cards")
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(DECK_CARDS_TABLE)
    )

    print(
        "Silver deck_cards "
        "delete-then-insert completed."
    )

else:
    print(
        "Nothing to replace in "
        f"{DECK_CARDS_TABLE}."
    )

In [0]:
display(
    spark.table(DECK_CARDS_TABLE)
    .orderBy(
        F.col("updated_at").desc()
    )
    .limit(50)
)